In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    strike_rate: float
    ball_per_boundary: float
    boundary_percentage: float
    summary: str

In [20]:
def StrikeRate(state: BatsmanState) -> BatsmanState:
  strike_rate = (state['runs']/state['balls'])*100
  return {'strike_rate': strike_rate}

In [21]:
def ballPerBoundary(state: BatsmanState) -> BatsmanState:
  ball_per_boundary = state['balls']/(state['fours'] + state['sixes'])
  return {'ball_per_boundary': ball_per_boundary}

In [22]:
def boundaryPercentage(state: BatsmanState) -> BatsmanState:
  boundary_percentage = ((state['fours'] + state['sixes'])/state['balls'])*100
  return {'boundary_percentage': boundary_percentage}

In [23]:
def Summary(state: BatsmanState) -> BatsmanState:
  summary = f"The batsman scored {state['runs']} runs in {state['balls']} balls with a strike rate of {state['strike_rate']:.2f}. He hit {state['fours']} fours and {state['sixes']} sixes, resulting in a boundary percentage of {state['boundary_percentage']:.2f}% and a ball per boundary ratio of {state['ball_per_boundary']:.2f}."
  return {'summary': summary}

In [24]:
#deine graph
graph = StateGraph(BatsmanState)

#add nodes
graph.add_node('Strike-Rate', StrikeRate)
graph.add_node('Ball-Per-Boundary', ballPerBoundary)
graph.add_node('Boundary-Percentage', boundaryPercentage)
graph.add_node('summary', Summary)

#add parallel edges
graph.add_edge(START, 'Strike-Rate')
graph.add_edge(START, 'Ball-Per-Boundary')
graph.add_edge(START, 'Boundary-Percentage')
graph.add_edge('Strike-Rate', 'summary')
graph.add_edge('Ball-Per-Boundary', 'summary')
graph.add_edge('Boundary-Percentage', 'summary')
graph.add_edge('summary', END)

#compile
workflow = graph.compile()

In [28]:
#execute
input_state = BatsmanState(runs=120, balls=100, fours=10, sixes=5)
output_state = workflow.invoke(input_state)
print(output_state['summary'])

The batsman scored 120 runs in 100 balls with a strike rate of 120.00. He hit 10 fours and 5 sixes, resulting in a boundary percentage of 15.00% and a ball per boundary ratio of 6.67.
